# 99 · Clean up

**The simplest way to remove everything is to delete the project:**

```bash
gcloud projects delete YOUR_PROJECT_ID
```

To keep the project, run the cells below in order: the orchestrator, the Compliance Agent, the Model Armor
template, then what notebook 00 created. Each Terraform destroy asks for nothing and cannot be undone.

In [ ]:
from nbsetup import *

## The orchestrator

Its Terraform state includes the engine, its telemetry and IAM. The engine goes first, on its own: it now holds
sessions and memories, and the API only deletes an engine with child resources when asked to `force`, which
Terraform can't do. `terraform destroy` then finds it gone and removes the rest. Deleting an engine can leave its
Agent Identity's IAM grants behind; the last cell lists any that remain.

In [ ]:
import time
import vertexai

client = vertexai.Client(project=PROJECT, location=REGION)
client.agent_engines.delete(name=engine(), force=True)
while True:  # deletion is asynchronous; wait until the engine is gone
    try:
        client.agent_engines.get(name=engine())
        time.sleep(10)
    except Exception:
        break
print("deleted", engine())

In [ ]:
%%bash
cd "$REPO/growth-risk-agent/deployment/terraform/single-project"
terraform init -input=false >/dev/null && terraform destroy -input=false -auto-approve 2>&1 | tail -4

In [ ]:
%%bash
gcloud projects get-iam-policy "$PROJECT" --format=json \
  | python3 -c "import json,sys; [print(b['role'], m) for b in json.load(sys.stdin)['bindings'] for m in b['members'] if 'reasoningEngines' in m]" \
  || true
echo "(any lines above are leftover grants: remove them with gcloud projects remove-iam-policy-binding)"

## The Compliance Agent

In [ ]:
%%bash
cd "$REPO/compliance-agent/deployment/terraform/single-project"
terraform init -input=false >/dev/null && terraform destroy -input=false -auto-approve 2>&1 | tail -4

## The Model Armor template

In [ ]:
%%bash
cd "$REPO/platform/terraform"
terraform init -input=false >/dev/null && terraform destroy -input=false -auto-approve 2>&1 | tail -4

## The setup resources

`setup/99_teardown.sh` deletes the MCP server, the RAG corpora, the BigQuery dataset, the caller service account
and the Terraform state bucket, and sets RAG Engine to *Unprovisioned* (which stops its billing). It asks you to
type the project ID, so run it in a terminal:

```bash
./setup/99_teardown.sh
```